In [2]:
import random
from collections import defaultdict, deque

def vecinos(r,c,n):
  for dr, dc in [(-1,0), (1,0), (0,-1), (0,1)]:
    nr, nc = r + dr, c + dc
    if 0 <= nr < n and 0 <= nc < n:
      yield  nr, nc


def ruta_directa(src, dst, n):
  r, c = src
  dr, dc = dst
  path = [(r, c)]
  while r != dr:
    r += 1 if r < dr else -1
    path.append((r,c))
  while c != dc:
    c += 1 if c < dc else -1
    path.append((r,c))
  return path


def ruta_valiant(src, dst, n):
  m = (random.randrange(n), random.randrange(n))
  return ruta_directa(src, m, n) + ruta_directa(m , dst, n)[1:]


def ruta_aleatoria_permutacion(src, dst, n):
  r, c = src
  dr, dc = dst
  path = [(r, c)]
  while (r,c) != (dst[0], dst[1]):
    opciones = []
    if r != dr: opciones.append((r + (1 if r < dr else -1), c))
    if c != dc: opciones.append((r, c+ (1 if c < dc else -1)))
    if not opciones:
        break
    r, c = random.choice(opciones)
    path.append((r, c))
  return path


class Paquete:
    def __init__(self, pid, path):
        self.pid      = pid
        self.path     = path
        self.idx      = 0        # posición actual en la ruta
        self.retardo  = 0        # pasos extra por congestión
        self.entregado = False

    @property
    def pos(self):
        return self.path[self.idx]

    @property
    def siguiente(self):
        return self.path[self.idx + 1] if self.idx + 1 < len(self.path) else None


def simular(paquetes, verbose=False):
    """
    Modelo de tiempo discreto con resolución de conflictos en aristas.
    En cada paso, a lo sumo UN paquete puede cruzar cada arista dirigida.
    Los paquetes que no pueden avanzar esperan (retardo +1).
    """
    paso = 0
    while not all(p.entregado for p in paquetes):
        paso += 1
        aristas_usadas = set()

        for pk in paquetes:
            if pk.entregado:
                continue
            sig = pk.siguiente
            if sig is None:
                pk.entregado = True
                if verbose:
                    print(f"  Paso {paso}: paquete {pk.pid} ENTREGADO "
                          f"(retardo acumulado: {pk.retardo})")
                continue

            arista = (pk.pos, sig)
            if arista not in aristas_usadas:
                aristas_usadas.add(arista)
                pk.idx += 1
            else:
                pk.retardo += 1   # congestión: esperar un paso

    return paso


def congestión_máxima(paquetes):
    """Cuenta cuántos paquetes comparten cada arista en toda su ruta."""
    conteo = defaultdict(int)
    for pk in paquetes:
        for i in range(len(pk.path) - 1):
            conteo[(pk.path[i], pk.path[i+1])] += 1
    return max(conteo.values(), default=0)


# ─────────────────────────────────────────────────────
# Comparación experimental
# ─────────────────────────────────────────────────────

def comparar(n=6, num_paquetes=20, trials=200):
    resultados = {'directo': [], 'valiant': [], 'sesgo_aleatorio': []}

    for _ in range(trials):
        pares = []
        for _ in range(num_paquetes):
            src = (random.randrange(n), random.randrange(n))
            dst = (random.randrange(n), random.randrange(n))
            while src == dst:
                dst = (random.randrange(n), random.randrange(n))
            pares.append((src, dst))

        for modo, fn in [('directo',         ruta_directa),
                         ('valiant',          ruta_valiant),
                         ('sesgo_aleatorio',  ruta_aleatoria_permutacion)]:
            pkts = [Paquete(i, fn(s, d, n)) for i, (s, d) in enumerate(pares)]
            cong = congestión_máxima(pkts)
            resultados[modo].append(cong)

    print(f"Malla {n}×{n} | {num_paquetes} paquetes | {trials} experimentos\n")
    print(f"{'Estrategia':<22} {'Cong. prom':>12} {'Cong. máx':>11} {'Cong. mín':>11}")
    print("─" * 58)
    for modo, vals in resultados.items():
        print(f"{modo:<22} {sum(vals)/len(vals):>12.2f} "
              f"{max(vals):>11} {min(vals):>11}")


# ─────────────────────────────────────────────────────
# Demo
# ─────────────────────────────────────────────────────

if __name__ == "__main__":
    random.seed(7)
    n, np_ = 5, 8

    pares = [((random.randrange(n), random.randrange(n)),
              (random.randrange(n), random.randrange(n)))
             for _ in range(np_)]
    pares = [(s, d) for s, d in pares if s != d]

    print("=== Ruteo directo ===")
    pkts_d = [Paquete(i, ruta_directa(s,d,n)) for i,(s,d) in enumerate(pares)]
    pasos_d = simular(pkts_d, verbose=True)
    retardo_d = sum(p.retardo for p in pkts_d)
    print(f"Pasos totales: {pasos_d} | Retardo acumulado: {retardo_d}\n")

    print("=== Ruteo Valiant (aleatorizado) ===")
    pkts_v = [Paquete(i, ruta_valiant(s,d,n)) for i,(s,d) in enumerate(pares)]
    pasos_v = simular(pkts_v, verbose=True)
    retardo_v = sum(p.retardo for p in pkts_v)
    print(f"Pasos totales: {pasos_v} | Retardo acumulado: {retardo_v}\n")

    print("─" * 58)
    comparar(n=6, num_paquetes=20, trials=300)


=== Ruteo directo ===
  Paso 2: paquete 2 ENTREGADO (retardo acumulado: 0)
  Paso 2: paquete 5 ENTREGADO (retardo acumulado: 0)
  Paso 2: paquete 6 ENTREGADO (retardo acumulado: 0)
  Paso 3: paquete 0 ENTREGADO (retardo acumulado: 0)
  Paso 3: paquete 1 ENTREGADO (retardo acumulado: 0)
  Paso 4: paquete 4 ENTREGADO (retardo acumulado: 0)
  Paso 7: paquete 3 ENTREGADO (retardo acumulado: 0)
  Paso 7: paquete 7 ENTREGADO (retardo acumulado: 0)
Pasos totales: 7 | Retardo acumulado: 0

=== Ruteo Valiant (aleatorizado) ===
  Paso 2: paquete 6 ENTREGADO (retardo acumulado: 0)
  Paso 4: paquete 2 ENTREGADO (retardo acumulado: 0)
  Paso 4: paquete 5 ENTREGADO (retardo acumulado: 0)
  Paso 5: paquete 1 ENTREGADO (retardo acumulado: 0)
  Paso 7: paquete 7 ENTREGADO (retardo acumulado: 0)
  Paso 9: paquete 3 ENTREGADO (retardo acumulado: 0)
  Paso 12: paquete 4 ENTREGADO (retardo acumulado: 0)
  Paso 13: paquete 0 ENTREGADO (retardo acumulado: 0)
Pasos totales: 13 | Retardo acumulado: 0

────────